# FaceTune — Hybrid AI Detector Training Notebook

**Pipeline:** FFT-ResNet50 + frozen CLIP ViT-B/32 → Fusion MLP → binary logit

| Experiment | Description |
|---|---|
| `hybrid_v1` | Frozen backbones, train fusion only (start here) |
| `hybrid_v2_hardneg` | Hard negatives + medium augmentation |
| `hybrid_v3_sd3fix` | SD3-targeted hard negatives + heavy augmentation |

Results saved to: `/content/drive/MyDrive/facetune_detector_runs/<experiment_name>/`

Each run produces: `config.yaml`, `metrics.json`, `train.log`, `confusion_matrix.png`, `predictions.csv`, `checkpoint.pt`

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
!pip install -q transformers datasets torchvision pyyaml matplotlib
print('Dependencies installed ✓')

In [ ]:
# ── 2. Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/facetune_detector_runs'
HF_CACHE_DIR = '/content/drive/MyDrive/hf_cache'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
print(f'Drive runs dir:  {DRIVE_DIR}')
print(f'HF cache dir:    {HF_CACHE_DIR}')

In [ ]:
# ── 3. Clone / pull repo ─────────────────────────────────────────────────
REPO = '/content/FaceTune'
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/alicesolov/FaceTune.git {REPO}

os.chdir(REPO)
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('Repo ready:', REPO)

In [ ]:
# ── 4. Select experiment ─────────────────────────────────────────────────
# Change this to switch between hybrid_v1 / hybrid_v2_hardneg / hybrid_v3_sd3fix
EXPERIMENT = 'hybrid_v1'

import yaml
with open(f'configs/{EXPERIMENT}.yaml') as f:
    cfg = yaml.safe_load(f)

print(f'Experiment: {EXPERIMENT}')
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ── 5. Load dataset ───────────────────────────────────────────────────────
from training.dataset import DatasetConfig, load_defactify_split

print('Loading train split ...')
train_hf = load_defactify_split(DatasetConfig(split='train', cache_dir=HF_CACHE_DIR))
print(f'  train: {len(train_hf)} rows')

print('Loading validation split ...')
val_hf = load_defactify_split(DatasetConfig(split='validation', cache_dir=HF_CACHE_DIR))
print(f'  val:   {len(val_hf)} rows')

In [ ]:
# ── 6. Build hybrid data loaders ─────────────────────────────────────────
from pathlib import Path
from datasets.hard_negatives import build_hybrid_loaders

data_cfg   = cfg.get('data', {})
phase1_cfg = cfg['training']['phase1']

# fp.json location (produced by diagnose step; only needed for v2/v3)
FP_JSON = Path(DRIVE_DIR) / 'fp.json'

train_loader, val_loader = build_hybrid_loaders(
    train_hf, val_hf,
    batch_size=phase1_cfg['batch_size'],
    augmentation=data_cfg.get('augmentation', False),
    augmentation_strength=data_cfg.get('augmentation_strength', 'medium'),
    hard_negatives=data_cfg.get('hard_negatives', False),
    fp_json=FP_JSON if data_cfg.get('hard_negatives') else None,
    hn_weight=data_cfg.get('hard_neg_weight', 2.5),
    num_workers=2,
)
print(f'Loaders: train={len(train_loader)} batches  val={len(val_loader)} batches')

In [ ]:
# ── 7. Build model ───────────────────────────────────────────────────────
import torch
from training.model_factory import build_model_from_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model_from_config(cfg).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: total={total:,}  trainable={trainable:,} ({100*trainable/total:.1f}%)')

# Quick forward-pass sanity check
shapes = model.embedding_shapes(device=device)
print('Embedding shapes:', shapes)

In [ ]:
# ── 8. Experiment logger ─────────────────────────────────────────────────
from training.experiment_logger import ExperimentLogger

logger = ExperimentLogger(EXPERIMENT, drive_dir=DRIVE_DIR)
logger.save_config(cfg)
CHECKPOINT_PATH = logger.checkpoint_path
print(f'Run dir:    {logger.exp_dir}')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Drive sync: {logger.drive_dir}')

In [ ]:
# ── 9. Phase 1: Train fusion only (frozen backbones) ─────────────────────
import time, torch.nn as nn
from training.train import (train_one_epoch_hybrid, validate_hybrid,
                             save_checkpoint, EarlyStopping)

phase1  = cfg['training']['phase1']
ckpt_by = cfg['checkpoint'].get('by', 'fpr')

optimizer = torch.optim.Adam(
    model.get_parameter_groups(
        lr_fusion=phase1['lr_fusion'],
        lr_backbone=phase1.get('lr_backbone', 0.0),
    )
)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
stopper   = EarlyStopping(patience=cfg['training'].get('early_stopping_patience', 3))

best_f1, best_fpr = 0.0, 1.0
print(f'=== Phase 1: {phase1["epochs"]} epochs — fusion only (backbones frozen) ===')

for epoch in range(1, phase1['epochs'] + 1):
    t0 = time.time()
    train_loss, train_m = train_one_epoch_hybrid(
        model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_m = validate_hybrid(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    print(f'[P1 E{epoch}/{phase1["epochs"]}]  '
          f'train f1={train_m["f1"]:.4f} fpr={train_m["fpr"]:.4f}  |  '
          f'val f1={val_m["f1"]:.4f} fpr={val_m["fpr"]:.4f}  [{elapsed:.0f}s]')

    logger.log_epoch(epoch, train_m, val_m,
                     extra={'phase': 1, 'train_loss': train_loss})

    improved = (ckpt_by == 'fpr' and val_m['fpr'] < best_fpr) or \
               (ckpt_by != 'fpr' and val_m['f1'] > best_f1)
    if improved:
        best_fpr = min(best_fpr, val_m['fpr'])
        best_f1  = max(best_f1,  val_m['f1'])
        save_checkpoint(model, epoch, val_m, CHECKPOINT_PATH)
        print(f'  ✓ checkpoint saved  (fpr={best_fpr:.4f}  f1={best_f1:.4f})')

    logger.sync_to_drive()
    stopper.step(val_m['f1'])
    if stopper.should_stop:
        print('Early stopping.'); break

print(f'Phase 1 done. Best FPR={best_fpr:.4f}  F1={best_f1:.4f}')

In [ ]:
# ── 10. Phase 2: Unfreeze FFT layer4 ─────────────────────────────────────
model.unfreeze_fft_last_block()
trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable parameters: {trainable2:,}')

phase2 = cfg['training']['phase2']
optimizer2 = torch.optim.Adam(
    model.get_parameter_groups(
        lr_fusion=phase2['lr_fusion'],
        lr_backbone=phase2['lr_backbone'],
    )
)
stopper2 = EarlyStopping(patience=cfg['training'].get('early_stopping_patience', 3))

print(f'=== Phase 2: {phase2["epochs"]} epochs — FFT layer4 unfrozen ===')

for epoch in range(1, phase2['epochs'] + 1):
    t0 = time.time()
    train_loss, train_m = train_one_epoch_hybrid(
        model, train_loader, optimizer2, criterion, device, scaler)
    val_loss, val_m = validate_hybrid(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    global_epoch = phase1['epochs'] + epoch
    print(f'[P2 E{epoch}/{phase2["epochs"]}]  '
          f'train f1={train_m["f1"]:.4f} fpr={train_m["fpr"]:.4f}  |  '
          f'val f1={val_m["f1"]:.4f} fpr={val_m["fpr"]:.4f}  [{elapsed:.0f}s]')

    logger.log_epoch(global_epoch, train_m, val_m,
                     extra={'phase': 2, 'train_loss': train_loss})

    improved = (ckpt_by == 'fpr' and val_m['fpr'] < best_fpr) or \
               (ckpt_by != 'fpr' and val_m['f1'] > best_f1)
    if improved:
        best_fpr = min(best_fpr, val_m['fpr'])
        best_f1  = max(best_f1,  val_m['f1'])
        save_checkpoint(model, global_epoch, val_m, CHECKPOINT_PATH)
        print(f'  ✓ checkpoint saved  (fpr={best_fpr:.4f}  f1={best_f1:.4f})')

    logger.sync_to_drive()
    stopper2.step(val_m['f1'])
    if stopper2.should_stop:
        print('Early stopping.'); break

print(f'Phase 2 done. Best FPR={best_fpr:.4f}  F1={best_f1:.4f}')

In [ ]:
# ── 11. Diagnose false positives (run after first experiment) ────────────
# Save fp.json to Drive for use in hybrid_v2_hardneg
import json
from training.diagnose import diagnose

diag_result = diagnose(
    weights_path=CHECKPOINT_PATH,
    split='validation',
    top_n=2000,
    cache_dir=HF_CACHE_DIR,
)

with open(FP_JSON, 'w') as f:
    json.dump(diag_result, f, indent=2)

print(f"FPR: {diag_result['fpr']:.1%}  ({diag_result['false_positives_total']} FP out of {diag_result['real_total']} real)")
print(f"Scene categories: {diag_result.get('scene_category_breakdown', {})}")
print(f"fp.json → {FP_JSON}")

In [ ]:
# ── 12. Evaluation on test split ─────────────────────────────────────────
import json
from collections import defaultdict
from torch.utils.data import DataLoader
from transformers import CLIPImageProcessor
from datasets.hard_negatives import DefactifyHybridDataset
from training.dataset import DatasetConfig, load_defactify_split
from training.model_factory import load_checkpoint_any
from training.train import compute_metrics
import torch.nn.functional as F

print('Loading test split ...')
test_hf  = load_defactify_split(DatasetConfig(split='test', cache_dir=HF_CACHE_DIR))
clip_proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-base-patch32')
test_ds  = DefactifyHybridDataset(test_hf, clip_proc)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

eval_model, meta = load_checkpoint_any(CHECKPOINT_PATH, device=device)
eval_model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.inference_mode():
    for fft_t, clip_t, labels in test_loader:
        logits = eval_model(fft_t.to(device), clip_t.to(device))
        probs  = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

metrics = compute_metrics(all_preds, all_labels)
print('\n=== Test results ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── 13. Per-generator recall ──────────────────────────────────────────────
gen_preds  = defaultdict(list)
gen_labels = defaultdict(list)

for i, item in enumerate(test_hf):
    gen = str(item.get('Label_B', 'unknown'))
    gen_preds[gen].append(all_preds[i])
    gen_labels[gen].append(all_labels[i])

print('\n=== Per-generator recall (test) ===')
per_gen_results = {}
for gen in sorted(gen_preds):
    ai_idx = [i for i, l in enumerate(gen_labels[gen]) if l == 1]
    if not ai_idx:
        continue
    recall = sum(gen_preds[gen][i] == 1 for i in ai_idx) / len(ai_idx)
    per_gen_results[gen] = round(recall, 4)
    print(f'  {gen:<15} n={len(ai_idx):>5}  recall={recall:.4f}')

In [ ]:
# ── 14. Save confusion matrix + predictions CSV ───────────────────────────
cm = [[0, 0], [0, 0]]
for p, l in zip(all_preds, all_labels):
    cm[l][p] += 1
logger.save_confusion_matrix(cm, class_names=['real', 'ai_generated'])

pred_rows = [
    {'index': i, 'label': all_labels[i], 'pred': all_preds[i],
     'p_ai': round(all_probs[i], 4)}
    for i in range(min(len(all_preds), 10000))
]
logger.save_predictions(pred_rows)
logger.sync_to_drive()

print(f'Confusion matrix:')
print(f'  True\\Pred   real    ai_gen')
print(f'  real       {cm[0][0]:>6}  {cm[0][1]:>6}')
print(f'  ai_gen     {cm[1][0]:>6}  {cm[1][1]:>6}')
print(f'\nAll artifacts saved to: {logger.drive_dir}')

In [ ]:
# ── 15. Comparison table ─────────────────────────────────────────────────
baselines = {
    'Base FFT (ResNet-50)': {'acc': 0.8715, 'f1': 0.9248, 'fpr': 0.5113},
    '+ Hard Negatives':     {'acc': 0.8182, 'f1': 0.8843, 'fpr': 0.2604},
}
baselines[f'Hybrid ({EXPERIMENT})'] = {
    'acc': metrics['accuracy'], 'f1': metrics['f1'], 'fpr': metrics['fpr']
}

print(f'\n{"Model":<30} {"Acc":>7} {"F1":>7} {"FPR":>7}')
print('-' * 55)
for name, m in baselines.items():
    print(f'{name:<30} {m["acc"]:>7.4f} {m["f1"]:>7.4f} {m["fpr"]:>7.4f}')

print(f'\nTargets: FPR < 0.15  F1 > 0.91  SD3 recall > 0.80')
fpr_ok = metrics['fpr'] < 0.15
f1_ok  = metrics['f1'] > 0.91
sd3_ok = per_gen_results.get('SD3', 0.0) > 0.80
print(f'FPR:  {"✓" if fpr_ok else "✗"} ({metrics["fpr"]:.4f})')
print(f'F1:   {"✓" if f1_ok  else "✗"} ({metrics["f1"]:.4f})')
print(f'SD3:  {"✓" if sd3_ok else "✗"} ({per_gen_results.get("SD3", 0):.4f})')